# Retrain DARE3D

Companion to `Run_dare3d_Prediction.ipynb`. Where the prediction notebook *runs* the
pre-trained models, this one **retrains** them on your own labelled data, in the same two
stages:

1. **Segmentation** - detect the division centre (the barycentre between the two daughter cells).
2. **Regression** - estimate the division-axis orientation and length.

**What it does**
- Lets you point at a labelled dataset and set the training hyper-parameters.
- Validates the expected data layout before launching anything.
- Trains segmentation, then regression, by driving DARE3D's own `dare3d/train.py` - the same
  script the high-level `train_eval.py` wrapper and the napari **DARE3D training** widget call -
  as a subprocess, streaming the logs here.
- Evaluates the two trained models with `dare3d/eval.py`.
- Shows where the checkpoints land and how to feed them back into the prediction notebook /
  napari plugin.

> **A CUDA GPU is required** for training (the Lightning backward pass). See `README.md`
> ("Training Models") for the full background - this notebook keeps to the commands.

**Run each cell from top to bottom** (Shift+Enter).

---

In [ ]:
from pathlib import Path
import os, sys, subprocess, copy
from shlex import quote

## 1) Prerequisites

- Install DARE3D in a conda env as described in `README.md` (install the CUDA build of PyTorch
  first, then `pip install -r requirements.txt`, then `pip install -e .`).
- You can run this notebook from **anywhere in the checkout** - the setup cell below locates the
  repository root (the folder with `setup.py` and the `dare3d/` package) and `chdir`s there so the
  relative paths resolve.
- A CUDA GPU is required for training.

> Training was developed on Linux/HPC. On some Windows setups the training compute can crash
> natively even though inference is fine; if that happens, train on Linux/HPC and use the
> resulting model directory with the prediction notebook / plugin.

## 2) Prepare your labelled dataset

DARE3D reads training data from `data/3d/<dataset>/{train,val}/{im,label}`, where every `.tif`
is a 4-D movie `(T, Z, Y, X)`:

```
data/3d/<dataset>/
|-- train/
|   |-- im/      *.tif      # raw movies (T, Z, Y, X)
|   `-- label/   *.tif      # labels: daughter 1 -> odd ids, daughter 2 -> even ids
`-- val/
    |-- im/      *.tif
    `-- label/   *.tif
```

An optional `train/weights/` folder gives per-voxel sparse weighting (defaults to all-ones).
See `README.md` -> "Train on Custom Dataset" for how labels encode the daughter-cell pairs, and
the `scripts/` helpers for splitting raw movies into this train/val layout.

In [ ]:
# ---- Dataset -------------------------------------------------------------
# Folder name under data/3d/ that holds the train/ and val/ splits (see the layout above).
dataset = "my_dataset"

# ---- Hyper-parameters (mirror the documented train_eval.py defaults) -----
epochs             = 50        # max epochs per stage
batch_size         = 8
cell_radius        = 8         # radius (voxels) of the segmentation target sphere
seg_crop_size      = 128       # segmentation patch size
date               = "retrain" # names the run folder: logs/<task>/runs/<date>/
threshold          = None      # segmentation threshold for eval; None = auto-search
train_segmentation = True
train_regression   = True

# Locate the repo root (folder with setup.py + dare3d/) by walking up, then chdir
# there so relative paths (dare3d/train.py, data/3d/...) resolve whether this
# notebook lives in notebooks/ or at the repo root.
def _find_repo_root(start: Path) -> Path:
    for d in (start, *start.parents):
        if (d / "setup.py").is_file() and (d / "dare3d").is_dir():
            return d
    raise SystemExit("Could not locate the DARE3D repo root (no setup.py found above this notebook).")

repo_root = _find_repo_root(Path.cwd().resolve())
os.chdir(repo_root)

# Keep MLflow's tracking store local and portable. SQLite works with mlflow>=3 (which dropped
# the file store); for older mlflow you may instead use:
#   mlflow_uri = "file:///" + (repo_root / "logs" / "mlflow" / "mlruns").as_posix()
(repo_root / "logs" / "mlflow").mkdir(parents=True, exist_ok=True)
mlflow_uri = "sqlite:///" + (repo_root / "logs" / "mlflow" / "mlflow.db").as_posix()

print("Repo root :", repo_root)
print("MLflow    :", mlflow_uri)

In [ ]:
data_root = repo_root / "data" / "3d" / dataset
errors = []
if not (repo_root / "dare3d" / "train.py").exists():
    errors.append("Cannot find dare3d/train.py - is this a DARE3D checkout?")
for split in ("train", "val"):
    for sub in ("im", "label"):
        d = data_root / split / sub
        n = len([p for p in d.iterdir() if p.suffix.lower() in (".tif", ".tiff")]) if d.is_dir() else 0
        if not d.is_dir():
            errors.append(f"Missing folder: {d}")
        elif n == 0:
            errors.append(f"No .tif files in: {d}")
        else:
            print(f"OK  {d}  ({n} .tif)")
if errors:
    print("\nSome paths are missing:\n- " + "\n- ".join(errors))
    raise SystemExit("Fix the dataset layout above and re-run this cell.")
print("\nDataset layout looks good.")

> **Voxel scale (optional).** Training isotropises the data using a per-movie scale in
> um/voxel, read from `data/3d/scales.json` (the segmentation/regression experiments fall back
> to a sensible default scale). Add an entry per movie if your voxels are anisotropic - see
> `README.md` -> "Register Your Data Scales".

## 3) Train the segmentation model

Launches `dare3d/train.py experiment=segmentation ...` and streams its output. Stop any time
with *Kernel -> Interrupt*. Checkpoints are written to `logs/segmentation3d_<dataset>/runs/<date>/`.

In [ ]:
def run(cmd):
    """Run a subprocess, streaming its output here. Interrupt the kernel to stop."""
    print("Command preview:\n" + " ".join(quote(str(c)) for c in cmd) + "\n")
    env = copy.copy(os.environ)
    env["HYDRA_FULL_ERROR"] = "1"
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, universal_newlines=True, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"Process exited with code {proc.returncode}")

seg_task = f"segmentation3d_{dataset}"
seg_cmd = [
    sys.executable, str(Path("dare3d") / "train.py"),
    "experiment=segmentation",
    f"task_name={seg_task}",
    f"train_dir=3d/{dataset}/train",
    f"val_dir=3d/{dataset}/val",
    f"test_dir=3d/{dataset}/val",
    "trainer.accelerator=gpu",
    f"data.batch_size={batch_size}",
    "data.num_workers=0",
    "steps_per_epoch=1000",
    f"trainer.max_epochs={epochs}",
    "model/criterion=dice_focal",
    "model.optimizer.lr=0.1",
    "model/scheduler=one_cycle_lr",
    "model.scheduler_interval='step'",
    "renorm='min-max'",
    "time_axis_padding=1",
    f"cell_radius={cell_radius}",
    f"date={date}",
    f"crop_size={seg_crop_size}",
    f"logger.mlflow.tracking_uri={mlflow_uri}",
]
if train_segmentation:
    run(seg_cmd)
else:
    print("Skipping segmentation (train_segmentation = False)")

## 4) Train the regression model

Same idea with `experiment=regression`. Checkpoints land in `logs/regression3d_<dataset>/runs/<date>/`.

In [ ]:
reg_task = f"regression3d_{dataset}"
reg_cmd = [
    sys.executable, str(Path("dare3d") / "train.py"),
    "experiment=regression",
    f"task_name={reg_task}",
    f"train_dir=3d/{dataset}/train",
    f"val_dir=3d/{dataset}/val",
    f"trainer.max_epochs={epochs}",
    f"date={date}",
    "steps_per_epoch=1000",
    "model.optimizer.lr=0.001",
    f"data.batch_size={batch_size}",
    "model/net=simple_regression_net",
    "model.net.n_stages=3",
    "model.net.start_filters=32",
    f"logger.mlflow.tracking_uri={mlflow_uri}",
]
if train_regression:
    run(reg_cmd)
else:
    print("Skipping regression (train_regression = False)")

## 5) Evaluate the trained models

Runs `dare3d/eval.py` on the two model directories (auto-picking the best `epoch_*.ckpt` if
present), mirroring what `train_eval.py` does after training.

In [ ]:
seg_model_dir = repo_root / "logs" / seg_task / "runs" / date
reg_model_dir = repo_root / "logs" / reg_task / "runs" / date

def find_epoch_ckpt(model_dir):
    ckpt_dir = model_dir / "checkpoints"
    if ckpt_dir.is_dir():
        for c in ckpt_dir.glob("*.ckpt"):
            if "epoch" in c.stem:
                return c.name
    return None

eval_cmd = [
    sys.executable, str(Path("dare3d") / "eval.py"),
    f"segmentation.model_dir={seg_model_dir}",
    f"regression.model_dir={reg_model_dir}",
    f"segmentation.threshold={'' if threshold is None else threshold}",
]
seg_ck = find_epoch_ckpt(seg_model_dir)
reg_ck = find_epoch_ckpt(reg_model_dir)
if seg_ck:
    eval_cmd.append(f"segmentation.ckpt_name={seg_ck}")
if reg_ck:
    eval_cmd.append(f"regression.ckpt_name={reg_ck}")
run(eval_cmd)

## 6) Use your retrained models

The trained model directories are:

- segmentation: `logs/segmentation3d_<dataset>/runs/<date>/`
- regression:   `logs/regression3d_<dataset>/runs/<date>/`

Each contains `checkpoints/` (`last.ckpt`, `epoch_*.ckpt`) and `.hydra/config.yaml` - exactly the
`model_dir` layout the prediction step expects. To run inference with them:

- **Notebook:** open `Run_dare3d_Prediction.ipynb` and set `segmentation_model_dir` /
  `regression_model_dir` to the two paths above.
- **napari plugin:** point the **segmentation / regression model dir** fields at them (the
  DARE3D training widget auto-fills these after a run).

Inspect the training curves with MLflow (using the `mlflow_uri` printed in step 2):

```bash
mlflow ui --backend-store-uri sqlite:///logs/mlflow/mlflow.db
```